In [3]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage,BaseMessage
import requests

In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational"
)

model = ChatHuggingFace(llm=llm)

/Users/harshraj/Desktop/Langchain Models/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Creating a sample tool

In [5]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [9]:
print(multiply.invoke({'a':3, 'b':4}))
print(multiply.args)

12
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Binding - Registering my custom tool to LLM

In [22]:
messages = []

In [23]:
modek = model.bind_tools([multiply])

In [26]:
query = HumanMessage("Product of 3 and 5")
messages.append(query)

In [30]:
ans = modek.invoke([query])

In [33]:
messages.append(ans)

In [35]:
tool_result = multiply.invoke(ans.tool_calls[0])
tool_result

ToolMessage(content='15', name='multiply', tool_call_id='5hPAioeC6')

In [36]:
messages.append(tool_result)

In [40]:
messages

[HumanMessage(content='Product of 3 and 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 5}', 'name': 'multiply', 'description': None}, 'id': '5hPAioeC6', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 87, 'total_tokens': 104}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e74b2-8508-7ca3-b045-78986bb44c54-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': '5hPAioeC6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 17, 'total_tokens': 104}),
 ToolMessage(content='15', name='multiply', tool_call_id='5hPAioeC6')]

In [42]:
finl = modek.invoke(messages).content
print(finl)

The product of 3 and 5 is 15.
